# jHealth: train the exercise model on Google's computers

Everything here runs on Google Colab, not on your laptop. Two public datasets, both read straight from Zenodo:

- **REHAB24-6** (video): squat, lunge, leg abduction, arm abduction, arm VW. Videos are fetched one at a time, turned
  into body points with MediaPipe, and deleted. Research use only (CC BY-NC 4.0).
- **UI-PRMD** (motion capture): sit-to-stand. Marker positions converted to the same body points. Public domain.

At the end your browser downloads small files: one trained model per exercise (under 1 MB each).

**How to run**
1. Open https://colab.research.google.com, then **File → Upload notebook** and choose this file (`ml/colab_train.ipynb`).
2. **Runtime → Run all.** If Colab warns that the notebook was not written by Google, choose *Run anyway*.
3. Wait. Squats with the heavy tracker take roughly 20 to 40 minutes on Colab's free computer.
4. Files such as `squat.json` and `sit_to_stand.json` download.
   - `squat.json` (and other video exercises): put it in `~/jhealth/web/public/models/` (the app uses it at once) and `~/jhealth/ml/models/`.
   - `sit_to_stand.json`: put it in `~/jhealth/ml/models/` only. It was trained on motion capture and must first be
     calibrated to the camera with `ml/calibrate_camera.py` and a few recorded sit-to-stand clips (no labels needed).

The settings cell below chooses the exercises and the tracker.

In [ ]:
# Install what Colab does not have yet (into this notebook's own Python).
import importlib.util
missing = [p for p in ("mediapipe", "remotezip") if importlib.util.find_spec(p) is None]
if missing:
    get_ipython().run_line_magic("pip", "install -q " + " ".join(missing))

In [ ]:
# ---- settings -----------------------------------------------------------------------------------------------
EXERCISES = [6]          # REHAB24-6 video exercises: 6 squat, 5 lunge, 4 leg abduction, 1 arm abduction, 2 arm VW ([] = none)
SIT_TO_STAND = True      # also train sit-to-stand from UI-PRMD (motion capture, fast: no videos to process)
POSE_MODEL = "heavy"     # "heavy" is slower and more accurate (+0.06 macro F1 on squats); "full" is the default model
STRIDE = 2               # look at every 2nd frame (15 per second): plenty for slow rehab movements
QUICK_TEST = False       # True: only 3 videos from one camera, to check that the notebook runs at all
import os
QUICK_TEST = QUICK_TEST or os.environ.get("JHEALTH_QUICK_TEST") == "1"
if QUICK_TEST:
    POSE_MODEL, STRIDE = "full", 6
UIPRMD_SUBJECTS = {1, 2, 3} if QUICK_TEST else None   # None = all 10 people
if os.environ.get("JHEALTH_EXERCISES") is not None:   # for testing: e.g. JHEALTH_EXERCISES="" = sit-to-stand only
    EXERCISES = [int(x) for x in os.environ["JHEALTH_EXERCISES"].split(",") if x]

## 1. The labels and the pose model

REHAB24-6 ([Zenodo](https://zenodo.org/records/13305826)): 10 healthy adults filmed in a lab; a physiotherapist marked
every repetition as correct or not. Licence CC BY-NC 4.0: research and prototypes only, not a commercial product.

In [ ]:
import csv, io, shutil, tempfile, time
import requests
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
import numpy as np
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, roc_auc_score

ZENODO = "https://zenodo.org/api/records/13305826/files"
WORK = Path(tempfile.mkdtemp(prefix="jhealth_"))
labels_csv = requests.get(f"{ZENODO}/Segmentation.csv/content", timeout=120).text
ROWS = [r for r in csv.DictReader(io.StringIO(labels_csv), delimiter=";") if int(r["exercise_id"]) in EXERCISES]
MODEL_PATH = WORK / f"pose_landmarker_{POSE_MODEL}.task"
if EXERCISES:
  MODEL_PATH.write_bytes(requests.get(
    "https://storage.googleapis.com/mediapipe-models/pose_landmarker/"
    f"pose_landmarker_{POSE_MODEL}/float16/latest/pose_landmarker_{POSE_MODEL}.task", timeout=300).content)

# Which frames each video needs: the labelled repetitions, merged into blocks, with 15 frames of margin.
spans, exercise_of = defaultdict(list), {}
for r in ROWS:
    spans[r["video_id"]].append((int(r["first_frame"]), int(r["last_frame"])))
    exercise_of[r["video_id"]] = int(r["exercise_id"])
BLOCKS = {}
for vid, s in spans.items():
    s.sort()
    merged = [[max(0, s[0][0] - 15), s[0][1] + 15]]
    for a, b in s[1:]:
        if a - 15 - merged[-1][1] <= 60:
            merged[-1][1] = b + 15
        else:
            merged.append([max(0, a - 15), b + 15])
    BLOCKS[vid] = merged
JOBS = [(v, cam) for v in sorted(BLOCKS) for cam in ((17,) if QUICK_TEST else (17, 18))][: 3 if QUICK_TEST else None]
print(f"{len(ROWS)} labelled repetitions, {len(JOBS)} videos to process, tracker: {POSE_MODEL}")

## 2. Videos → body points (the slow part)

Each video (10 to 40 MB) is pulled out of the 2.6 GB archive on its own, run through MediaPipe, and deleted.
Nothing is kept except the body points.

In [ ]:
def track(job):
    import cv2, mediapipe as mp
    from mediapipe.tasks import python as mp_python
    from mediapipe.tasks.python import vision
    from remotezip import RemoteZip
    vid, cam = job
    suffix = "Camera17-30fps.mp4" if cam == 17 else "Camera18-30fps-transposed.mp4"
    path = WORK / f"{vid}_{cam}.mp4"
    for attempt in range(5):                       # Zenodo drops connections now and then
        try:
            with RemoteZip(f"{ZENODO}/videos.zip/content") as z, z.open(f"Ex{exercise_of[vid]}/{vid}-{suffix}") as src, open(path, "wb") as dst:
                shutil.copyfileobj(src, dst)
            break
        except Exception:
            if attempt == 4:
                raise
            time.sleep(5 * (attempt + 1))
    options = vision.PoseLandmarkerOptions(base_options=mp_python.BaseOptions(model_asset_path=str(MODEL_PATH)),
                                           running_mode=vision.RunningMode.VIDEO, num_poses=1)
    cap = cv2.VideoCapture(str(path))
    frames, world, idx = [], [], 0
    for start, end in BLOCKS[vid]:
        lm = vision.PoseLandmarker.create_from_options(options)
        while idx < start and cap.grab():
            idx += 1
        while idx <= end:
            if (idx - start) % STRIDE:
                if not cap.grab():
                    break
                idx += 1
                continue
            ok, bgr = cap.read()
            if not ok:
                break
            res = lm.detect_for_video(mp.Image(image_format=mp.ImageFormat.SRGB, data=cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)), int(idx * 1000 / 30))
            frames.append(idx)
            world.append([(p.x, p.y, p.z) for p in res.pose_world_landmarks[0]] if res.pose_world_landmarks else np.full((33, 3), np.nan))
            idx += 1
        lm.close()
    cap.release()
    path.unlink(missing_ok=True)
    return vid, cam, np.array(frames), np.array(world, dtype=np.float32)

STREAMS, t0 = {}, time.time()
with ThreadPoolExecutor(max_workers=2 if QUICK_TEST else max(1, min(4, os.cpu_count() or 1))) as pool:
    for n, (vid, cam, fr, w) in enumerate(pool.map(track, JOBS), 1):
        STREAMS[(vid, cam)] = (fr, w)
        print(f"[{n}/{len(JOBS)} {time.time() - t0:4.0f}s] {vid} camera {cam}: {len(fr)} frames")

## 3. Body points → repetitions

Each labelled repetition is cut out and stretched to exactly 64 frames, like `build_dataset.py` does.

In [ ]:
def fill_gaps(x):
    bad = np.isnan(x[:, 0, 0])
    if bad.any() and not bad.all():
        i = np.arange(len(x)); flat = x.reshape(len(x), -1)
        for j in range(flat.shape[1]):
            flat[bad, j] = np.interp(i[bad], i[~bad], flat[~bad, j])
    return x

def resample(x, n=64):
    src, dst = np.linspace(0, 1, len(x)), np.linspace(0, 1, n)
    flat = x.reshape(len(x), -1)
    return np.stack([np.interp(dst, src, flat[:, j]) for j in range(flat.shape[1])], 1).reshape(n, *x.shape[1:])

reps = defaultdict(list)
for r in ROWS:
    for cam in (17, 18):
        if (r["video_id"], cam) not in STREAMS:
            continue
        fr, w = STREAMS[(r["video_id"], cam)]
        m = (fr >= int(r["first_frame"])) & (fr <= int(r["last_frame"]))
        if m.sum() < 10 or np.isnan(w[m][:, 0, 0]).mean() > 0.2:
            continue
        reps["lm"].append(resample(fill_gaps(w[m].astype(np.float64))).astype(np.float32))
        reps["exercise"].append(int(r["exercise_id"])); reps["person"].append(int(r["person_id"]))
        reps["mistake"].append(1 - int(r["correctness"]))
        reps["duration"].append((int(r["last_frame"]) - int(r["first_frame"]) + 1) / 30)
REPS = {k: np.array(v) for k, v in reps.items()}
print({k: v.shape for k, v in REPS.items()} if REPS else "no REHAB24-6 exercises chosen")

## 3b. Sit-to-stand from UI-PRMD (motion capture)

The Vicon marker positions are converted into the same 33 body points MediaPipe gives (shoulders, elbows, wrists,
knees and ankles are markers; the hip joints are estimated from the pelvis markers, Bell et al. 1990), turned so
that "up" matches MediaPipe, and stretched to 64 frames. Each file is one repetition: standing, sitting, standing.
The code is copied from `ml/uiprmd.py`; `python uiprmd.py --check` shows the conversion behaves like a real body.

Caveat, measured: a model trained on motion capture loses a lot on camera body points. On REHAB24-6 squats
(which have both) it fell from 0.83 to 0.55 macro F1; putting the camera data on its own scale won back part of it
(0.69). So the score below is the motion-capture score, and `ml/calibrate_camera.py` must calibrate the model with a
few camera clips before the app uses it (`ml/transfer_check.py` repeats the measurement).

In [ ]:
UIPRMD = 'https://zenodo.org/api/records/23104965/files'
VICON_HZ = 100
MARKER = {'LFHD': 1, 'RFHD': 2, 'LSHO': 10, 'LELB': 12, 'LWRA': 14, 'LWRB': 15, 'RSHO': 17, 'RELB': 19, 'RWRA': 21, 'RWRB': 22, 'LASI': 24, 'RASI': 25, 'LPSI': 26, 'RPSI': 27, 'LKNE': 29, 'LANK': 31, 'LHEE': 32, 'RKNE': 35, 'RANK': 37, 'RHEE': 38}


def vicon_to_mediapipe(pos_mm):
    """(frames, 117) Vicon marker positions in mm -> (frames, 33, 3) in metres, MediaPipe's indices, y pointing down."""
    P = np.asarray(pos_mm, dtype=np.float64).reshape(len(pos_mm), 39, 3) / 1000.0
    g = lambda k: P[:, MARKER[k] - 1]  # noqa: E731
    # Hip joint centres (Bell et al., 1990): from the middle of the two front pelvis markers (ASIS), 30% of the
    # inter-ASIS width down, 14% back and 36% sideways.
    mid_asi = (g("LASI") + g("RASI")) / 2
    mid_psi = (g("LPSI") + g("RPSI")) / 2
    lateral = g("RASI") - g("LASI")
    width = np.linalg.norm(lateral, axis=1, keepdims=True)
    lateral = lateral / width
    anterior = mid_asi - mid_psi
    anterior -= (anterior * lateral).sum(1, keepdims=True) * lateral
    anterior /= np.linalg.norm(anterior, axis=1, keepdims=True)
    superior = np.cross(lateral, anterior)
    head = (g("LFHD") + g("RFHD")) / 2
    if ((head - mid_asi) * superior).sum(1).mean() < 0:
        superior = -superior          # "up" must point at the head, whatever the lab's axes
    hip = lambda side: mid_asi + width * (-0.30 * superior - 0.14 * anterior + side * 0.36 * lateral)  # noqa: E731
    out = np.full((len(P), 33, 3), np.nan)
    for i, v in {0: head, 11: g("LSHO"), 12: g("RSHO"), 13: g("LELB"), 14: g("RELB"),
                 15: (g("LWRA") + g("LWRB")) / 2, 16: (g("RWRA") + g("RWRB")) / 2,
                 23: hip(-1), 24: hip(+1), 25: g("LKNE"), 26: g("RKNE"), 27: g("LANK"), 28: g("RANK"),
                 29: g("LHEE"), 30: g("RHEE")}.items():
        out[:, i] = v
    # Turn the lab frame so that "up" (ankles -> head) is -y, as in MediaPipe's world landmarks (a proper rotation).
    up = (head - (g("LANK") + g("RANK")) / 2).mean(0)
    up /= np.linalg.norm(up)
    target = np.array([0.0, -1.0, 0.0])
    v = np.cross(up, target)
    s, c = np.linalg.norm(v), up @ target
    K = np.array([[0, -v[2], v[1]], [v[2], 0, -v[0]], [-v[1], v[0], 0]])
    R = np.eye(3) + K + K @ K * ((1 - c) / s ** 2) if s > 1e-8 else (np.eye(3) if c > 0 else np.diag([1.0, -1.0, -1.0]))
    return out @ R.T


def fill_marker_gaps(pos_mm, max_missing=0.2):
    """Markers the cameras lost for a few frames (missing or exactly 0,0,0) are filled in by interpolating over time.
    Returns None if a marker we need is missing in more than `max_missing` of the frames."""
    P = np.asarray(pos_mm, dtype=np.float64).reshape(len(pos_mm), 39, 3).copy()
    missing = ~np.isfinite(P).all(-1) | (np.abs(np.nan_to_num(P)).sum(-1) == 0)
    used = [m - 1 for m in MARKER.values()]
    if missing[:, used].mean(0).max() > max_missing:
        return None
    t = np.arange(len(P))
    for j in used:
        bad = missing[:, j]
        if bad.any():
            for c in range(3):
                P[bad, j, c] = np.interp(t[bad], t[~bad], P[~bad, j, c])
    return P.reshape(len(pos_mm), -1)


def fetch_sit_to_stand(subjects=None):
    """Read UI-PRMD's sit-to-stand Vicon repetitions straight from Zenodo.
    Returns a list of (landmarks (frames, 33, 3), mistake 0/1, subject, seconds)."""
    import re
    from remotezip import RemoteZip
    reps, skipped = [], []
    for archive, mistake in (("Segmented Movements.zip", 0), ("Incorrect Segmented Movements.zip", 1)):
        with RemoteZip(f"{UIPRMD}/{archive.replace(' ', '%20')}/content") as z:
            for name in z.namelist():
                m = re.search(r"/Vicon/Positions/m05_s(\d+)_e(\d+)_positions", name)
                if not m or (subjects and int(m.group(1)) not in subjects):
                    continue
                raw = np.loadtxt(z.read(name).decode().splitlines(), delimiter=",")
                pos = fill_marker_gaps(raw)
                if pos is None:
                    skipped.append(name.rsplit("/", 1)[-1])
                    continue
                reps.append((vicon_to_mediapipe(pos), mistake, int(m.group(1)), len(pos) / VICON_HZ))
    if skipped:
        print(f"skipped {len(skipped)} repetitions with markers missing in over 20% of frames: {skipped}")
    return reps

In [ ]:
STS = None
if SIT_TO_STAND:
    t0 = time.time()
    raw = fetch_sit_to_stand(UIPRMD_SUBJECTS)
    STS = {
        "lm": np.stack([resample(lm.astype(np.float64)).astype(np.float32) for lm, _, _, _ in raw]),
        "mistake": np.array([m for _, m, _, _ in raw]),
        "person": np.array([s for _, _, s, _ in raw]),
        "duration": np.array([d for _, _, _, d in raw], dtype=np.float32),
    }
    print(f"sit-to-stand: {len(raw)} repetitions from {len(set(STS['person']))} people in {time.time() - t0:.0f}s")

## 4. The features (copied from `train_model.ipynb`)

Angles per frame, then 11 numbers per angle curve. Identical to the notebook and to the browser (`web/src/lib/pose/features.ts`).

In [ ]:
# MediaPipe numbers its 33 body points. These are the ones we use.
L_SHOULDER, R_SHOULDER, L_ELBOW, R_ELBOW, L_WRIST, R_WRIST = 11, 12, 13, 14, 15, 16
L_HIP, R_HIP, L_KNEE, R_KNEE, L_ANKLE, R_ANKLE = 23, 24, 25, 26, 27, 28
LEFT = dict(sh=L_SHOULDER, el=L_ELBOW, wr=L_WRIST, hip=L_HIP, knee=L_KNEE, ankle=L_ANKLE)
RIGHT = dict(sh=R_SHOULDER, el=R_ELBOW, wr=R_WRIST, hip=R_HIP, knee=R_KNEE, ankle=R_ANKLE)
DOWN = np.array([0.0, 1.0, 0.0], dtype=np.float32)      # in MediaPipe's world, y points at the floor

# Index list that swaps every left point with its right twin (used for mirroring).
SWAP = np.arange(33)
for a, b in [(1, 4), (2, 5), (3, 6), (7, 8), (9, 10)] + [(i, i + 1) for i in range(11, 33, 2)]:
    SWAP[a], SWAP[b] = b, a

def length(v):
    return np.linalg.norm(v, axis=-1)

def unit(v):
    return v / (length(v)[..., None] + 1e-9)

def angle_between(u, v):
    "Angle between two directions, in degrees."
    return np.degrees(np.arccos(np.clip((unit(u) * unit(v)).sum(-1), -1, 1)))

def joint_angle(p, a, b, c):
    "Angle at point b, between the lines b->a and b->c. A straight limb gives 180."
    return angle_between(p[..., a, :] - p[..., b, :], p[..., c, :] - p[..., b, :])

def side_signals(p, s, forward):
    "Measurements for one side of the body. p has shape (..., 33, 3)."
    thigh = p[..., s["knee"], :] - p[..., s["hip"], :]
    shin = p[..., s["knee"], :] - p[..., s["ankle"], :]
    upper_arm = p[..., s["el"], :] - p[..., s["sh"], :]
    arm_len = length(upper_arm) + length(p[..., s["wr"], :] - p[..., s["el"], :])
    return {
        "knee":       joint_angle(p, s["hip"], s["knee"], s["ankle"]),   # 180 = straight leg
        "hip":        joint_angle(p, s["sh"], s["hip"], s["knee"]),      # 180 = standing tall
        "thigh_incl": angle_between(thigh, DOWN),                        # 0 = thigh points straight down
        "thigh_fwd":  (unit(thigh) * forward).sum(-1),                   # above 0 = thigh swings forward
        "knee_fwd":   (shin * forward).sum(-1) / (length(shin) + 1e-9),  # above 0 = knee in front of the ankle
        "shoulder":   joint_angle(p, s["hip"], s["sh"], s["el"]),        # 0 = arm by the side, 180 = overhead
        "elbow":      joint_angle(p, s["sh"], s["el"], s["wr"]),         # 180 = straight arm
        "wrist_h":    (p[..., s["sh"], 1] - p[..., s["wr"], 1]) / (arm_len + 1e-9),   # above 0 = wrist over the shoulder
        "arm_fwd":    (unit(upper_arm) * forward).sum(-1),               # above 0 = arm goes forward, not sideways
    }

def body_signals(p):
    "Measurements that use both sides at once."
    mid_sh = (p[..., L_SHOULDER, :] + p[..., R_SHOULDER, :]) / 2
    mid_hip = (p[..., L_HIP, :] + p[..., R_HIP, :]) / 2
    mid_ankle = (p[..., L_ANKLE, :] + p[..., R_ANKLE, :]) / 2
    sh_line = p[..., R_SHOULDER, :] - p[..., L_SHOULDER, :]
    hip_line = p[..., R_HIP, :] - p[..., L_HIP, :]
    leg_len = sum(length(p[..., s["knee"], :] - p[..., s["hip"], :]) + length(p[..., s["knee"], :] - p[..., s["ankle"], :])
                  for s in (LEFT, RIGHT)) / 2
    return {
        "trunk_lean":    angle_between(mid_sh - mid_hip, -DOWN),                     # 0 = upright
        "shoulder_tilt": np.degrees(np.arcsin(np.clip(unit(sh_line)[..., 1], -1, 1))),   # 0 = shoulders level
        "pelvis_tilt":   np.degrees(np.arcsin(np.clip(unit(hip_line)[..., 1], -1, 1))),  # 0 = hips level
        "knee_gap":      length(p[..., R_KNEE, :] - p[..., L_KNEE, :]) / (length(hip_line) + 1e-9),   # under 1 = knees closer than hips
        "ankle_gap":     length(p[..., R_ANKLE, :] - p[..., L_ANKLE, :]) / (length(hip_line) + 1e-9), # stance width
        "hip_h":         (mid_ankle[..., 1] - mid_hip[..., 1]) / (leg_len + 1e-9),   # about 1 = standing, smaller = lower
        "wrist_gap":     length(p[..., R_WRIST, :] - p[..., L_WRIST, :]) / (length(sh_line) + 1e-9),
        "thigh_spread":  angle_between(p[..., L_KNEE, :] - p[..., L_HIP, :], p[..., R_KNEE, :] - p[..., R_HIP, :]),
    }

def forward_axis(p):
    "Unit vector, level with the floor, pointing where the chest points."
    right = unit(p[..., R_SHOULDER, :] - p[..., L_SHOULDER, :])     # from the left shoulder to the right one
    return unit(np.cross(-DOWN, right))

# For one-sided exercises: how we tell which side is working.
#   arm / leg abduction: the side whose arm / thigh swings through the bigger range
#   lunge: the front leg is the one whose thigh points forward
WORKING_SIDE_BY = {1: ("shoulder", "range"), 4: ("thigh_incl", "range"), 5: ("thigh_fwd", "mean")}
# A leg exercise is judged on legs and trunk, an arm exercise on arms and trunk.
AREA = {6: "legs", 5: "legs", 4: "legs", 1: "arms", 2: "arms"}
PAIRED = {"legs": ["knee", "hip", "thigh_incl", "thigh_fwd", "knee_fwd"],
          "arms": ["shoulder", "elbow", "wrist_h", "arm_fwd"]}
SINGLE = {"legs": ["trunk_lean", "shoulder_tilt", "pelvis_tilt", "knee_gap", "ankle_gap", "hip_h", "thigh_spread"],
          "arms": ["trunk_lean", "shoulder_tilt", "pelvis_tilt", "wrist_gap"]}

def left_is_working(p, ex):
    "True for reps where the left side moves more than the right. p: (reps, frames, 33, 3)."
    f = forward_axis(p)
    key, how = WORKING_SIDE_BY[ex]
    l, r = side_signals(p, LEFT, f)[key], side_signals(p, RIGHT, f)[key]
    return np.ptp(l, axis=1) > np.ptp(r, axis=1) if how == "range" else l.mean(axis=1) > r.mean(axis=1)

def mirror(p):
    "Swap left and right, as if looking at the person in a mirror."
    q = p[..., SWAP, :].copy()
    q[..., 0] *= -1
    return q

def signals(p, ex):
    "(reps, frames, 33, 3) -> names, array (reps, signals, frames)"
    p = p.copy()
    one_sided = ex in WORKING_SIDE_BY
    if one_sided:
        flip = left_is_working(p, ex)
        p[flip] = mirror(p[flip])
    f = forward_axis(p)
    left, right, body = side_signals(p, LEFT, f), side_signals(p, RIGHT, f), body_signals(p)
    names, series = [], []
    for k in PAIRED[AREA[ex]]:
        if one_sided:   # working side and resting side
            names += [f"work_{k}", f"rest_{k}"]; series += [right[k], left[k]]
        else:           # both sides do the same: keep the average and the left-right difference
            names += [f"avg_{k}", f"diff_{k}"]; series += [(left[k] + right[k]) / 2, np.abs(left[k] - right[k])]
    for k in SINGLE[AREA[ex]]:
        names.append(k)
        series.append(body[k] if one_sided or "tilt" not in k else np.abs(body[k]))
    return names, np.stack(series, axis=1).astype(np.float32)

In [ ]:
STATS = ["min", "max", "range", "mean", "std", "when_min", "when_max", "speed", "peak_speed", "jerk", "first_vs_second_half"]

def summarise(sig, dur):
    "(reps, signals, frames) -> (reps, signals * 11 + 1)"
    n = sig.shape[-1]
    speed = np.abs(np.diff(sig, axis=-1)) * n                          # change per whole repetition
    parts = [sig.min(-1), sig.max(-1), np.ptp(sig, -1), sig.mean(-1), sig.std(-1),
             sig.argmin(-1) / n, sig.argmax(-1) / n,                   # when the lowest / highest value happens
             speed.mean(-1), speed.max(-1),                            # average and fastest movement
             np.abs(np.diff(sig, n=2, axis=-1)).mean(-1),              # jerkiness
             sig[..., :n // 2].mean(-1) - sig[..., n // 2:].mean(-1)]  # going down versus coming back up
    return np.concatenate([np.stack(parts, -1).reshape(len(sig), -1), dur[:, None]], axis=1)

In [ ]:
def leave_one_person_out(make_model, X, y, who):
    "Returns, for every sample, the mistake probability from a model that never saw that person."
    prob = np.zeros(len(y))
    for p in np.unique(who):
        hidden = who == p
        model = make_model().fit(X[~hidden], y[~hidden])
        prob[hidden] = model.predict_proba(X[hidden])[:, 1]
    return prob

def scores(y, prob):
    pred = (prob >= 0.5).astype(int)
    return {"accuracy": accuracy_score(y, pred), "F1 (mistake)": f1_score(y, pred, zero_division=0),
            "macro F1": f1_score(y, pred, average="macro", zero_division=0),
            "AUC": roc_auc_score(y, prob) if prob.std() > 0 else 0.5}

## 5. Models, tested honestly

Leave one person out: every score below is measured on a person the model never saw. Choosing the best of several
on the same test flatters it slightly; your own recorded clips are the fair check.

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler


def logistic(C=0.1):
    return make_pipeline(StandardScaler(), LogisticRegression(C=C, max_iter=4000, class_weight="balanced"))


def forest():
    return RandomForestClassifier(n_estimators=300, min_samples_leaf=2, class_weight="balanced", n_jobs=2, random_state=0)


def boosting():
    return HistGradientBoostingClassifier(max_depth=3, learning_rate=0.05, max_iter=200, class_weight="balanced", random_state=0)


class Ensemble:
    """Average of the members' mistake probabilities. Simple, and usually steadier than any single member."""
    def __init__(self, makers):
        self.makers = makers

    def fit(self, X, y):
        self.members = [m().fit(X, y) for m in self.makers]
        return self

    def predict_proba(self, X):
        p = np.mean([m.predict_proba(X)[:, 1] for m in self.members], axis=0)
        return np.stack([1 - p, p], axis=1)


def loso(make, X, y, who, extra=None):
    """Leave one person out. `extra` = (X, y, who) added to training only, never containing the hidden person."""
    prob = np.zeros(len(y))
    for p in np.unique(who):
        hidden = who == p
        Xt, yt = X[~hidden], y[~hidden]
        if extra is not None:
            keep = extra[2] != p
            Xt, yt = np.vstack([Xt, extra[0][keep]]), np.r_[yt, extra[1][keep]]
        prob[hidden] = make().fit(Xt, yt).predict_proba(X[hidden])[:, 1]
    return prob


def to_dict(model):
    if isinstance(model, Ensemble):
        return {"type": "ensemble", "members": [to_dict(m) for m in model.members]}
    if isinstance(model, HistGradientBoostingClassifier):
        trees = []
        for (predictor,) in model._predictors:
            n = predictor.nodes
            trees.append({"feature": n["feature_idx"].tolist(), "threshold": n["num_threshold"].tolist(),
                          "left": n["left"].tolist(), "right": n["right"].tolist(),
                          "leaf": n["is_leaf"].astype(int).tolist(), "value": n["value"].tolist()})
        return {"type": "boosting", "baseline": float(np.ravel(model._baseline_prediction)[0]), "trees": trees}
    if isinstance(model, RandomForestClassifier):
        trees = []
        for est in model.estimators_:
            t = est.tree_
            v = t.value[:, 0, :]
            trees.append({"feature": t.feature.tolist(), "threshold": t.threshold.tolist(), "left": t.children_left.tolist(),
                          "right": t.children_right.tolist(), "p_mistake": (v[:, 1] / v.sum(1)).tolist()})
        return {"type": "forest", "trees": trees}
    if hasattr(model, "named_steps"):
        sc, lr = model[0], model[1]
        return {"type": "logistic", "mean": sc.mean_.tolist(), "scale": sc.scale_.tolist(),
                "weights": lr.coef_[0].tolist(), "bias": float(lr.intercept_[0])}
    raise ValueError("this model type cannot be exported to the browser")


def predict_from_dict(m, x):
    """The exported model evaluated with plain arithmetic: what the browser does (web/src/lib/pose/model.ts)."""
    if m["type"] == "ensemble":
        return float(np.mean([predict_from_dict(k, x) for k in m["members"]]))
    if m["type"] == "logistic":
        z = m["bias"] + sum(w * (v - mu) / sd for w, v, mu, sd in zip(m["weights"], x, m["mean"], m["scale"]))
        return 1 / (1 + np.exp(-z))
    if m["type"] == "boosting":
        raw = m["baseline"]
        for t in m["trees"]:
            node = 0
            while not t["leaf"][node]:
                node = t["left"][node] if x[t["feature"][node]] <= t["threshold"][node] else t["right"][node]
            raw += t["value"][node]
        return 1 / (1 + np.exp(-raw))
    total = 0.0
    for t in m["trees"]:
        node = 0
        while t["left"][node] != -1:
            node = t["left"][node] if x[t["feature"][node]] <= t["threshold"][node] else t["right"][node]
        total += t["p_mistake"][node]
    return total / len(m["trees"])

In [ ]:
NAMES = {1: "arm abduction", 2: "arm VW", 4: "leg abduction", 5: "lunge", 6: "squat"}
# Each task: name, repetitions, which REHAB24-6 exercise's feature settings to use (sit-to-stand: like the squat,
# both legs, not one-sided), where the data came from, and which tracker the browser must use with it.
TASKS = []
for e in EXERCISES:
    m = REPS["exercise"] == e if REPS else np.zeros(0, bool)
    TASKS.append((NAMES[e], {k: v[m] for k, v in REPS.items()} if REPS else {}, e,
                  "REHAB24-6 (CC BY-NC 4.0, research use only)", POSE_MODEL))
if STS is not None:
    TASKS.append(("sit to stand", STS, 6, "UI-PRMD Vicon motion capture (public domain, ODC PDDL 1.0)", "heavy"))
CANDIDATES = {
    "logistic regression": lambda: logistic(0.1),
    "random forest": forest,
    "gradient boosting": boosting,
    "ensemble: logistic + forest + boosting": lambda: Ensemble([lambda: logistic(0.1), forest, boosting]),
}
BEST = {}
for task, data, e, source, pose in TASKS:
    if not data or len(set(data["person"])) < 3:
        print(f"{task}: not enough people for an honest test, skipped"); continue
    names, sig = signals(data["lm"], e)
    X = summarise(sig, data["duration"].astype(np.float32))
    y, who = data["mistake"], data["person"]
    print(f"\n{task}: {len(y)} samples, {len(set(who))} people, {y.mean():.0%} with a mistake")
    results = {}
    for name, make in CANDIDATES.items():
        s = scores(y, loso(make, X, y, who))
        results[name] = s
        print(f"  {name:42s} macro F1 {s['macro F1']:.2f}   AUC {s['AUC']:.2f}")
    best = max(results, key=lambda k: (round(results[k]["macro F1"], 3), results[k]["AUC"]))
    BEST[task] = (best, results[best], X, y, who, names, e, source, pose)
    print(f"  best: {best}")

## 6. Export for the app

Trains the best model on everyone, checks that the exported file reproduces scikit-learn, and downloads it.

In [ ]:
import json
for task, (best, s, X, y, who, names, e, source, pose) in BEST.items():
    final = CANDIDATES[best]().fit(X, y)
    body = to_dict(final)
    gap = max(abs(predict_from_dict(body, row) - p) for row, p in zip(X[:60], final.predict_proba(X[:60])[:, 1]))
    assert gap < 1e-6, f"export does not reproduce scikit-learn ({gap})"
    spec = {"exercise": task, "model": best, "body_area": AREA[e], "one_sided": e in WORKING_SIDE_BY,
            "working_side_by": list(WORKING_SIDE_BY.get(e, [])), "pose_model": pose, "frames": 64,
            "signals": names, "stats": STATS, "features": [f"{n}:{st}" for n in names for st in STATS] + ["duration"],
            "tested_on_unseen_people": {k: round(float(v), 3) for k, v in s.items()},
            "training": {"samples": int(len(y)), "people": int(len(set(who))), "dataset": source},
            **body}
    out = Path(f"{task.replace(' ', '_')}.json")
    out.write_text(json.dumps(spec))
    print(f"saved {out} ({out.stat().st_size / 1e3:.0f} kB): {best}, macro F1 {s['macro F1']:.2f} on unseen people")
    try:
        from google.colab import files
        files.download(str(out))
    except ImportError:
        pass                                     # not on Colab: the file is in the current folder
shutil.rmtree(WORK, ignore_errors=True)